In [1]:
# Install packages if needed
# Uncomment if running on a fresh environment

# !pip install -q pandas numpy sacrebleu bert-score easse nltk bart-score


In [3]:
import sys, subprocess

def pip_install(pkg):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        print(f"✅ Installed: {pkg}")
    except subprocess.CalledProcessError:
        print(f"❌ Failed: {pkg}")

packages = [
    "sacrebleu",
    "bert-score",
    "easse",
    "nltk"
]

for pkg in packages:
    pip_install(pkg)

✅ Installed: sacrebleu
✅ Installed: bert-score
✅ Installed: easse
✅ Installed: nltk


In [24]:
!pip install git+https://github.com/feralvam/easse.git

  Cloning https://github.com/feralvam/easse.git to /tmp/pip-req-build-s2_pmphw
  Running command git clone --filter=blob:none --quiet https://github.com/feralvam/easse.git /tmp/pip-req-build-s2_pmphw
  Resolved https://github.com/feralvam/easse.git to commit 6a4352ec299ed03fda8ee45445ca43d9c7673e89
  Preparing metadata (setup.py) ... done
  Cloning https://github.com/facebookresearch/text-simplification-evaluation.git (to revision main) to /tmp/pip-install-_glp0hzn/tseval_d4558ec0484f4c0c9f9176015e6e797f
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/text-simplification-evaluation.git /tmp/pip-install-_glp0hzn/tseval_d4558ec0484f4c0c9f9176015e6e797f
  Resolved https://github.com/facebookresearch/text-simplification-evaluation.git to commit dea8863683ea5946fd50184883c9be7a7339e821
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 42.6 MB/s eta 0:00:00
  

In [9]:
import os
import re
import numpy as np
import pandas as pd
import sacrebleu

from bert_score import score as bert_score_fn
from nltk.translate.meteor_score import meteor_score
import nltk

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

# Try importing corpus_sari from easse first, then fall back to a simple implementation if unavailable
try:
    from easse.sari import corpus_sari
    HAS_EASSE = True
except Exception:
    HAS_EASSE = False
    print("easse not found. Install with: !pip install easse")
    corpus_sari = None

# BARTScore
try:
    import torch
    from bart_score import BARTScorer
    HAS_BARTSCORE = True
except Exception:
    HAS_BARTSCORE = False
    print("bart-score not found. Install with: !pip install bart-score")

bart-score not found. Install with: !pip install bart-score


## 1) Configure paths and column names

Edit only this cell.


In [1]:
# ===== File path =====
CSV_PATH = "/kaggle/input/datasets/naghamabdullahalshab/samples/final_df_with_lexical_simplifications.csv"  

# ===== Core columns in your CSV =====
COL_ORIGINAL = "Original_Sentence"
COL_REF1 = "level_1"
COL_REF2 = "level_2"
COL_REF3 = "level_3"
COL_LEXICAL = "lexical_simplified_text"

# ===== Optional hybrid prediction columns =====
# If your CSV already has hybrid outputs, put their names here.
# If they do not exist, leave them as None and the notebook will skip hybrid evaluation.
COL_HYBRID_L1 = None   # e.g. "hybrid_l1"
COL_HYBRID_L2 = None   # e.g. "hybrid_l2"
COL_HYBRID_L3 = None   # e.g. "hybrid_l3"

# ===== Runtime options =====
RUN_BERTSCORE = True
RUN_BARTSCORE = True
DEVICE = "cuda"  # or "cpu"


In [2]:
import pandas as pd

encodings = ["utf-8-sig", "utf-16", "utf-16le", "utf-16be", "cp1256", "windows-1256"]

for enc in encodings:
    try:
        df_test = pd.read_csv(CSV_PATH, encoding=enc, engine="python", nrows=5)
        print("✅ Encoding works:", enc)
        display(df_test.head())
        break
    except Exception as e:
        print("❌ Failed:", enc, "|", type(e).__name__, str(e)[:100])

✅ Encoding works: utf-8-sig


,Original_Sentence,level_1,level_2,level_3,evaluation_status,issue_details,cwi_details,complex_words,mlm_substitutions,filtered_substitutions,final_substitutions,lexical_simplified_text,replacement_map
0,"•• ""عدم الانحياز"" اصطلاح سياسي يعنى الحياد..","""عدم الانحياز"" يعني أن تكون محايداً ولا تفضل أ...","""عدم الانحياز"" هو مصطلح سياسي يعني أن تكون محا...","""عدم الانحياز"" هو مصطلح سياسي يشير إلى الحياد.",Pass,NaN,"[{'token': 'عدم', 'lemma': 'عَدَم', 'pos': 'no...","['الانحياز', 'اصطلاح', 'سياسي', 'الحياد']","[{'original_sentence': '•• ""عدم الانحياز"" اصطل...","[{'complex_word': 'الانحياز', 'target_pos': 'n...","[{'complex_word': 'الانحياز', 'replacement': '...","•• ""عدم الرضا"" مصطلح عربي يعنى الاستقلال..","['الانحياز -> الرضا', 'اصطلاح -> مصطلح', 'سياس..."
1,حيث تلتزم الدول غير المنحازة بعدم الارتباط بأي...,الدول غير المنحازة لا تتعامل مع دول الشرق أو ا...,تلتزم الدول غير المنحازة بعدم التحالف مع دول ا...,تلتزم الدول غير المنحازة بالحياد وعدم التحالف ...,Pass,NaN,"[{'token': 'حيث', 'lemma': 'حَيْثُ', 'pos': 'c...",['المنحازة'],[{'original_sentence': 'حيث تلتزم الدول غير ال...,"[{'complex_word': 'المنحازة', 'target_pos': 'n...",[],حيث تلتزم الدول غير المنحازة بعدم الارتباط بأي...,[]
2,بمعنى أن الدول غير المنحازة لا تقف موقف المتفر...,الدول غير المنحازة ليسوا متفرجين، هم يتدخلون و...,الدول غير المنحازة ليس لها موقف محايد في القضا...,بمعنى أن الدول غير المنحازة ليس موقفها موقف مت...,Moderate,Order Violation,"[{'token': 'بمعنى', 'lemma': 'مَعْنَى', 'pos':...","['المنحازة', 'المتفرج', 'العالمية']",[{'original_sentence': 'بمعنى أن الدول غير الم...,"[{'complex_word': 'المنحازة', 'target_pos': 'n...","[{'complex_word': 'المنحازة', 'replacement': '...",بمعنى أن الدول غير العظمى لا تقف موقف واحد أما...,"['المنحازة -> العظمى', 'المتفرج -> واحد', 'الع..."
3,وقد بدأ استعمال اصطلاح (عدم الانحياز) في مؤتمر...,"وقد بدأ استخدام مصطلح ""عدم الانحياز"" في مؤتمر ...","وقد بدأ استخدام مصطلح ""عدم الانحياز"" في مؤتمر ...","وقد بدأ استعمال اصطلاح ""عدم الانحياز"" في مؤتمر...",Moderate,Order Violation,"[{'token': 'وقد', 'lemma': 'قَدْ', 'pos': 'par...","['اصطلاح', 'الانحياز', '،', 'منحازة']",[{'original_sentence': 'وقد بدأ استعمال اصطلاح...,"[{'complex_word': 'اصطلاح', 'target_pos': 'nou...","[{'complex_word': 'اصطلاح', 'replacement': 'مص...",وقد بدأ استعمال مصطلح (عدم الاستقلال) في مؤتمر...,"['اصطلاح -> مصطلح', 'الانحياز -> الاستقلال', '..."
4,••السيارات التي يستعملها الدبلوماسيون.. أي رجا...,"السيارات التي يستخدمها الدبلوماسيون تسمى ""هيئة...",تتحمل السيارات التي يستخدمها الدبلوماسيون جملة...,تتحمل السيارات الدبلوماسية، أي تلك التي يستخدم...,Moderate,Order Violation,"[{'token': 'السيارات', 'lemma': 'سَيّارَة', 'p...",['سياسية'],[{'original_sentence': '••السيارات التي يستعمل...,"[{'complex_word': 'سياسية', 'target_pos': 'adj...","[{'complex_word': 'سياسية', 'replacement': 'ال...",••السيارات التي يستعملها الدبلوماسيون.. أي رجا...,['سياسية -> الدبلوماسية']


In [3]:
import pandas as pd
import csv
import sys

csv.field_size_limit(sys.maxsize)

df = pd.read_csv(
    CSV_PATH,
    encoding="utf-8-sig",
    engine="python",
    on_bad_lines="skip"
)

print("Shape:", df.shape)
print("Columns:", list(df.columns))
display(df.head())

Shape: (34294, 13)
Columns: ['Original_Sentence', 'level_1', 'level_2', 'level_3', 'evaluation_status', 'issue_details', 'cwi_details', 'complex_words', 'mlm_substitutions', 'filtered_substitutions', 'final_substitutions', 'lexical_simplified_text', 'replacement_map']


,Original_Sentence,level_1,level_2,level_3,evaluation_status,issue_details,cwi_details,complex_words,mlm_substitutions,filtered_substitutions,final_substitutions,lexical_simplified_text,replacement_map
0,"•• ""عدم الانحياز"" اصطلاح سياسي يعنى الحياد..","""عدم الانحياز"" يعني أن تكون محايداً ولا تفضل أ...","""عدم الانحياز"" هو مصطلح سياسي يعني أن تكون محا...","""عدم الانحياز"" هو مصطلح سياسي يشير إلى الحياد.",Pass,NaN,"[{'token': 'عدم', 'lemma': 'عَدَم', 'pos': 'no...","['الانحياز', 'اصطلاح', 'سياسي', 'الحياد']","[{'original_sentence': '•• ""عدم الانحياز"" اصطل...","[{'complex_word': 'الانحياز', 'target_pos': 'n...","[{'complex_word': 'الانحياز', 'replacement': '...","•• ""عدم الرضا"" مصطلح عربي يعنى الاستقلال..","['الانحياز -> الرضا', 'اصطلاح -> مصطلح', 'سياس..."
1,حيث تلتزم الدول غير المنحازة بعدم الارتباط بأي...,الدول غير المنحازة لا تتعامل مع دول الشرق أو ا...,تلتزم الدول غير المنحازة بعدم التحالف مع دول ا...,تلتزم الدول غير المنحازة بالحياد وعدم التحالف ...,Pass,NaN,"[{'token': 'حيث', 'lemma': 'حَيْثُ', 'pos': 'c...",['المنحازة'],[{'original_sentence': 'حيث تلتزم الدول غير ال...,"[{'complex_word': 'المنحازة', 'target_pos': 'n...",[],حيث تلتزم الدول غير المنحازة بعدم الارتباط بأي...,[]
2,بمعنى أن الدول غير المنحازة لا تقف موقف المتفر...,الدول غير المنحازة ليسوا متفرجين، هم يتدخلون و...,الدول غير المنحازة ليس لها موقف محايد في القضا...,بمعنى أن الدول غير المنحازة ليس موقفها موقف مت...,Moderate,Order Violation,"[{'token': 'بمعنى', 'lemma': 'مَعْنَى', 'pos':...","['المنحازة', 'المتفرج', 'العالمية']",[{'original_sentence': 'بمعنى أن الدول غير الم...,"[{'complex_word': 'المنحازة', 'target_pos': 'n...","[{'complex_word': 'المنحازة', 'replacement': '...",بمعنى أن الدول غير العظمى لا تقف موقف واحد أما...,"['المنحازة -> العظمى', 'المتفرج -> واحد', 'الع..."
3,وقد بدأ استعمال اصطلاح (عدم الانحياز) في مؤتمر...,"وقد بدأ استخدام مصطلح ""عدم الانحياز"" في مؤتمر ...","وقد بدأ استخدام مصطلح ""عدم الانحياز"" في مؤتمر ...","وقد بدأ استعمال اصطلاح ""عدم الانحياز"" في مؤتمر...",Moderate,Order Violation,"[{'token': 'وقد', 'lemma': 'قَدْ', 'pos': 'par...","['اصطلاح', 'الانحياز', '،', 'منحازة']",[{'original_sentence': 'وقد بدأ استعمال اصطلاح...,"[{'complex_word': 'اصطلاح', 'target_pos': 'nou...","[{'complex_word': 'اصطلاح', 'replacement': 'مص...",وقد بدأ استعمال مصطلح (عدم الاستقلال) في مؤتمر...,"['اصطلاح -> مصطلح', 'الانحياز -> الاستقلال', '..."
4,••السيارات التي يستعملها الدبلوماسيون.. أي رجا...,"السيارات التي يستخدمها الدبلوماسيون تسمى ""هيئة...",تتحمل السيارات التي يستخدمها الدبلوماسيون جملة...,تتحمل السيارات الدبلوماسية، أي تلك التي يستخدم...,Moderate,Order Violation,"[{'token': 'السيارات', 'lemma': 'سَيّارَة', 'p...",['سياسية'],[{'original_sentence': '••السيارات التي يستعمل...,"[{'complex_word': 'سياسية', 'target_pos': 'adj...","[{'complex_word': 'سياسية', 'replacement': 'ال...",••السيارات التي يستعملها الدبلوماسيون.. أي رجا...,['سياسية -> الدبلوماسية']


## 2) Clean rows and normalize text


In [4]:
import re

def normalize_unicode(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = text.replace("أ", "ا").replace("إ", "ا").replace("آ", "ا").replace("ٱ", "ا")
    text = text.replace("ى", "ي").replace("ؤ", "و").replace("ئ", "ي")
    text = re.sub(r"[\u064B-\u065F\u0670\u0617-\u061A\u06D6-\u06ED]", "", text)  # remove diacritics
    text = re.sub(r"\s+", " ", text).strip()
    return text

def clean_text_col(series):
    return series.fillna("").astype(str).map(normalize_unicode)

needed_cols = [COL_ORIGINAL, COL_REF1, COL_REF2, COL_REF3, COL_LEXICAL]
present_needed = [c for c in needed_cols if c is not None]

df = df.dropna(subset=present_needed).copy()

for col in [COL_ORIGINAL, COL_REF1, COL_REF2, COL_REF3, COL_LEXICAL]:
    df[col] = clean_text_col(df[col])

for col in [COL_HYBRID_L1, COL_HYBRID_L2, COL_HYBRID_L3]:
    if col is not None and col in df.columns:
        df[col] = clean_text_col(df[col])

print("Usable rows:", len(df))
df[[COL_ORIGINAL, COL_REF1, COL_REF2, COL_REF3, COL_LEXICAL]].head()

Usable rows: 34294


,Original_Sentence,level_1,level_2,level_3,lexical_simplified_text
0,"•• ""عدم الانحياز"" اصطلاح سياسي يعني الحياد..","""عدم الانحياز"" يعني ان تكون محايدا ولا تفضل احدا.","""عدم الانحياز"" هو مصطلح سياسي يعني ان تكون محا...","""عدم الانحياز"" هو مصطلح سياسي يشير الي الحياد.","•• ""عدم الرضا"" مصطلح عربي يعني الاستقلال.."
1,حيث تلتزم الدول غير المنحازة بعدم الارتباط باي...,الدول غير المنحازة لا تتعامل مع دول الشرق او ا...,تلتزم الدول غير المنحازة بعدم التحالف مع دول ا...,تلتزم الدول غير المنحازة بالحياد وعدم التحالف ...,حيث تلتزم الدول غير المنحازة بعدم الارتباط باي...
2,بمعني ان الدول غير المنحازة لا تقف موقف المتفر...,الدول غير المنحازة ليسوا متفرجين، هم يتدخلون و...,الدول غير المنحازة ليس لها موقف محايد في القضا...,بمعني ان الدول غير المنحازة ليس موقفها موقف مت...,بمعني ان الدول غير العظمي لا تقف موقف واحد اما...
3,وقد بدا استعمال اصطلاح (عدم الانحياز) في موتمر...,"وقد بدا استخدام مصطلح ""عدم الانحياز"" في موتمر ...","وقد بدا استخدام مصطلح ""عدم الانحياز"" في موتمر ...","وقد بدا استعمال اصطلاح ""عدم الانحياز"" في موتمر...",وقد بدا استعمال مصطلح (عدم الاستقلال) في موتمر...
4,••السيارات التي يستعملها الدبلوماسيون.. اي رجا...,"السيارات التي يستخدمها الدبلوماسيون تسمي ""هيية...",تتحمل السيارات التي يستخدمها الدبلوماسيون جملة...,تتحمل السيارات الدبلوماسية، اي تلك التي يستخدم...,••السيارات التي يستعملها الدبلوماسيون.. اي رجا...


In [19]:
!pip install -q textstat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 21.4 MB/s eta 0:00:00


## 4) Metric helpers


In [7]:
def osman_score(hypothesis, reference):
    """
    OSMAN approximation:
    Arabic-normalized METEOR average x 100.
    """
    scores = []
    for hyp, ref in zip(hypothesis, reference):
        hyp_norm = normalize_unicode(hyp)
        ref_norm = normalize_unicode(ref)
        hyp_tokens = hyp_norm.split()
        ref_tokens = ref_norm.split()
        if not hyp_tokens or not ref_tokens:
            scores.append(0.0)
            continue
        score = meteor_score([ref_tokens], hyp_tokens)
        scores.append(score)
    return float(np.mean(scores)) * 100

def safe_corpus_sari(orig_sents, sys_sents, refs_sents):
    if corpus_sari is None:
        raise ImportError("SARI requires easse. Please run: !pip install easse")
    return corpus_sari(
        orig_sents=orig_sents,
        sys_sents=sys_sents,
        refs_sents=refs_sents,
    )

def evaluate_predictions(sources, predictions, references, label, compute_sari=False):
    """
    Evaluate a list of predictions against one reference list.
    """
    print(f"\n{'='*70}")
    print(label)
    print(f"{'='*70}")

    result = {"label": label, "n": len(predictions)}

    if compute_sari:
        sari = safe_corpus_sari(
            orig_sents=sources,
            sys_sents=predictions,
            refs_sents=[references],
        )
        print(f"SARI        : {sari:.2f}")
        result["sari"] = sari

    bleu_score = sacrebleu.corpus_bleu(predictions, [references]).score
    print(f"BLEU        : {bleu_score:.2f}")
    result["bleu"] = bleu_score

    if RUN_BERTSCORE:
        P, R, F1 = bert_score_fn(
            predictions,
            references,
            lang="ar",
            verbose=False,
        )
        bs = F1.mean().item()
        print(f"BERTScore F1: {bs:.4f}")
        result["bertscore_f1"] = bs

    if RUN_BARTSCORE:
        bart_scores = bart_scorer.score(predictions, references, batch_size=8)
        bart_avg = float(np.mean(bart_scores))
        print(f"BARTScore   : {bart_avg:.4f}")
        result["bartscore"] = bart_avg

    osman = osman_score(predictions, references)
    print(f"OSMAN       : {osman:.2f}")
    result["osman"] = osman

    return result


## 5) Build lists from CSV


In [15]:
sources = df[COL_ORIGINAL].tolist()

ref1 = df[COL_REF1].tolist()
ref2 = df[COL_REF2].tolist()
ref3 = df[COL_REF3].tolist()

lexical_output = df[COL_LEXICAL].tolist()

hybrid_l1 = df[COL_HYBRID_L1].tolist() if (COL_HYBRID_L1 is not None and COL_HYBRID_L1 in df.columns) else None
hybrid_l2 = df[COL_HYBRID_L2].tolist() if (COL_HYBRID_L2 is not None and COL_HYBRID_L2 in df.columns) else None
hybrid_l3 = df[COL_HYBRID_L3].tolist() if (COL_HYBRID_L3 is not None and COL_HYBRID_L3 in df.columns) else None

print("Rows:", len(sources))


Rows: 34294


## 6) Compute the requested lexical SARI scores

Requested:
- `SARI(original, lexical_output, [ref1])`
- `SARI(original, lexical_output, [ref2])`
- `SARI(original, lexical_output, [ref3])`

This cell also computes BLEU, BERTScore, BARTScore, and OSMAN for the same lexical output vs each reference.


In [17]:
RUN_BERTSCORE = False
RUN_BARTSCORE = False

In [18]:
all_results = []

for refs, label in [
    (ref1, "Lexical output vs Ref 1"),
    (ref2, "Lexical output vs Ref 2"),
    (ref3, "Lexical output vs Ref 3"),
]:
    res = evaluate_predictions(
        sources=sources,
        predictions=lexical_output,
        references=refs,
        label=label,
        compute_sari=True,
    )
    all_results.append(res)

lexical_results_df = pd.DataFrame(all_results)
lexical_results_df


Lexical output vs Ref 1
SARI        : 17.40


BLEU        : 7.93
OSMAN       : 26.09

Lexical output vs Ref 2
SARI        : 22.59


BLEU        : 18.34
OSMAN       : 39.55

Lexical output vs Ref 3
SARI        : 24.81


BLEU        : 23.78
OSMAN       : 47.17


,label,n,sari,bleu,osman
0,Lexical output vs Ref 1,34294,17.399099,7.934071,26.086146
1,Lexical output vs Ref 2,34294,22.592245,18.343522,39.554145
2,Lexical output vs Ref 3,34294,24.814693,23.780851,47.170764


In [9]:
!pip install bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.7 MB/s eta 0:00:00


In [10]:
from bert_score import score as bert_score_fn
import pandas as pd

COL_LEXICAL = "lexical_simplified_text"

refs = {
    "Ref Level 1": "level_1",
    "Ref Level 2": "level_2",
    "Ref Level 3": "level_3"
}

all_results = []

for label, ref_col in refs.items():
    P, R, F1 = bert_score_fn(
        cands=df[COL_LEXICAL].fillna("").astype(str).tolist(),
        refs=df[ref_col].fillna("").astype(str).tolist(),
        lang="ar",
        verbose=True
    )

    all_results.append({
        "Comparison": f"Lexical vs {label}",
        "BERTScore_P": P.mean().item(),
        "BERTScore_R": R.mean().item(),
        "BERTScore_F1": F1.mean().item()
    })

bert_results_df = pd.DataFrame(all_results)
bert_results_df

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

...

  0%|          | 0/1070 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/536 [00:00<?, ?it/s]

done in 4249.12 seconds, 8.07 sentences/sec


,Comparison,BERTScore_P,BERTScore_R,BERTScore_F1
0,Lexical vs Ref Level 1,0.765473,0.802212,0.782685
1,Lexical vs Ref Level 2,0.813176,0.837670,0.824568
2,Lexical vs Ref Level 3,0.837453,0.858588,0.847264


In [13]:
import pandas as pd
from IPython.display import FileLink, display

final_scores_df = pd.DataFrame([
    {
        "label": "Lexical output vs Ref 1",
        "n": 34294,
        "SARI": 17.399099,
        "BLEU": 7.934071,
        "OSMAN_reference_based": 26.086146,
        "BERTScore_P": 0.765473,
        "BERTScore_R": 0.802212,
        "BERTScore_F1": 0.782685,
        "Average_OSMAN_readability": 88.89052716161507
    },
    {
        "label": "Lexical output vs Ref 2",
        "n": 34294,
        "SARI": 22.592245,
        "BLEU": 18.343522,
        "OSMAN_reference_based": 39.554145,
        "BERTScore_P": 0.813176,
        "BERTScore_R": 0.837670,
        "BERTScore_F1": 0.824568,
        "Average_OSMAN_readability": 88.89052716161507
    },
    {
        "label": "Lexical output vs Ref 3",
        "n": 34294,
        "SARI": 24.814693,
        "BLEU": 23.780851,
        "OSMAN_reference_based": 47.170764,
        "BERTScore_P": 0.837453,
        "BERTScore_R": 0.858588,
        "BERTScore_F1": 0.847264,
        "Average_OSMAN_readability": 88.89052716161507
    }
])

output_path = "/kaggle/working/lexical_final_scores_summary.csv"
final_scores_df.to_csv(output_path, index=False)

print(final_scores_df)
display(FileLink(output_path))

                     label      n       SARI       BLEU  \
0  Lexical output vs Ref 1  34294  17.399099   7.934071   
1  Lexical output vs Ref 2  34294  22.592245  18.343522   
2  Lexical output vs Ref 3  34294  24.814693  23.780851   

   OSMAN_reference_based  BERTScore_P  BERTScore_R  BERTScore_F1  \
0              26.086146     0.765473     0.802212      0.782685   
1              39.554145     0.813176     0.837670      0.824568   
2              47.170764     0.837453     0.858588      0.847264   

   Average_OSMAN_readability  
0                  88.890527  
1                  88.890527  
2                  88.890527  


/kaggle/working/lexical_final_scores_summary.csv